In [ ]:
import re
import json
import random
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/LW2')
CODE_DIR = PROJECT_DIR / 'code'
CODE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(CODE_DIR)

sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'LW1'))

from LW1.learn import *

# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)

EMB_DIR = Path("embeddings")
MODEL_DIR = Path("models_lr2")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EMB_DIM = 300
MIN_SEN_LEN = 6
MIN_WORD_LEN = 3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def load_embeddings():
    vocab, index_to_word = loadVocab()
    W, C = loadEmbeddings(EMB_DIM)
    return vocab, index_to_word, W

vocab, index_to_word, embedding_matrix = load_embeddings()

In [ ]:
assert embedding_matrix.shape[0] == len(vocab), "Размер vocab не совпадает с embedding_matrix"
assert embedding_matrix.shape[1] == EMB_DIM, "Размерность эмбеддингов не совпадает с EMB_DIM"

# if "<PAD>" not in vocab:
#     PAD_ID = len(vocab)
#     index_to_word[PAD_ID] = "<PAD>"
#     vocab["<PAD>"] = PAD_ID

#     pad_vector = np.zeros((1, EMB_DIM), dtype=embedding_matrix.dtype)
#     embedding_matrix = np.vstack([embedding_matrix, pad_vector])
# else:
#     PAD_ID = int(vocab["<PAD>"])

VOCAB_SIZE = len(vocab)

print("Размер словаря для ЛР2:", VOCAB_SIZE)
print("Форма embedding_matrix:", embedding_matrix.shape)

def read_raw_text(path):
    with open(path, 'r', encoding='cp1251') as file:
        text = file.read().lower()

    # Оставляем буквы, пробелы и знаки-разделители
    text = re.sub(r'[^а-яё\s–—\.!?;:,]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text


def parser_for_LSTM(path="../data"):
    text = ''

    for name in [str(i) for i in range(1, 5)]:
        text += read_raw_text(f'{path}/{name}.txt') + ' '

    raw_sentences = re.split(r'[–—\.!?;:,]+', text)
    print("Количество фрагментов после разбиения:", len(raw_sentences))

    parsed_sentences = []

    for sentence in raw_sentences:
        words = sentence.strip().split()

        # Удаляем короткие слова уже здесь
        words = [word for word in words if len(word) >= MIN_WORD_LEN]

        if len(words) >= MIN_SEN_LEN:
            parsed_sentences.append(words)

    return parsed_sentences


def prepare_pairs(sentences : list):
    X = []
    Y = []

    for sentence in sentences:
        for i in range(len(sentence) - MIN_SEN_LEN + 1):
            X.append(sentence[i : i + MIN_SEN_LEN - 1])
            Y.append(sentence[i + 1 : i + MIN_SEN_LEN])


    return X, Y

parsed_sentences = parser_for_LSTM()
X, Y = prepare_pairs(parsed_sentences)

print("Количество окон до перевода в индексы:", len(X))
print("\nПримеры:")

for ind in range(100, 110):
    print("X:", X[ind])
    print("Y:", Y[ind])
    print()

def pair_to_indices(X, Y, vocab, val_split=0.1, test_split=0.1):
    pairs = []
    skipped = 0

    for x, y in zip(X, Y):
        if all(word in vocab for word in x) and all(word in vocab for word in y):
            x_ids = [vocab[word] for word in x]
            y_ids = [vocab[word] for word in y]
            pairs.append((x_ids, y_ids))
        else:
            skipped += 1

    print("Всего пар до фильтрации:", len(X))
    print("Пропущено из-за слов вне vocab:", skipped)
    print("Осталось пар:", len(pairs))

    random.shuffle(pairs)

    X_ind = np.array([p[0] for p in pairs], dtype=np.int32)
    Y_ind = np.array([p[1] for p in pairs], dtype=np.int32)

    n = len(X_ind)

    test_size = int(n * test_split)
    val_size = int(n * val_split)

    X_test = X_ind[:test_size]
    Y_test = Y_ind[:test_size]

    X_val = X_ind[test_size:test_size + val_size]
    Y_val = Y_ind[test_size:test_size + val_size]

    X_train = X_ind[test_size + val_size:]
    Y_train = Y_ind[test_size + val_size:]

    return X_train, Y_train, X_val, Y_val, X_test, Y_test

X_train, Y_train, X_val, Y_val, X_test, Y_test = pair_to_indices(X, Y, vocab)

print("max X index:", X_train.max())
print("max Y index:", Y_train.max())
print("VOCAB_SIZE:", VOCAB_SIZE)

assert X_train.max() < VOCAB_SIZE
assert Y_train.max() < VOCAB_SIZE

print("\nФормы массивов:")
print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("X_val:", X_val.shape)
print("Y_val:", Y_val.shape)
print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)

print("\nПример индексов:")
print("X_train[0]:", X_train[0])
print("Y_train[0]:", Y_train[0])

from tensorflow.keras.layers import Embedding, Dense
from tensorflow.keras.models import Sequential

def make_model():
    embedding_layer = Embedding(input_dim=VOCAB_SIZE, output_dim=EMB_DIM, input_length=MIN_SEN_LEN - 1, trainable=False)
    embedding_layer.build((1,))
    embedding_layer.set_weights([embedding_matrix.astype("float32")])

    input = keras.Input(shape=(None,), dtype="int32")
    embedded = embedding_layer(input)
    _, state_h, state_c = layers.LSTM(256, return_state=True)(embedded)
    x = layers.LSTM(256, return_sequences=True)(embedded, initial_state=[state_h, state_c])
    x = layers.Dense(VOCAB_SIZE, activation="softmax")(x)
    model = keras.Model(input, x)
    model.summary()

    return model

model = make_model()

model.compile(loss=keras.losses.SparseCategoricalCrossentropy(), optimizer="adam", metrics=["acc"])

model.fit(X_train, Y_train, batch_size=64, epochs=10, validation_data=(X_val, Y_val))

model.save("../saved_results/model_2.keras")



Размер словаря для ЛР2: 66344
Форма embedding_matrix: (66344, 300)
Количество фрагментов после разбиения: 187811
Количество окон до перевода в индексы: 59699

Примеры:
X: ['нужно', 'быть', 'как', 'можно', 'неприметнее']
Y: ['быть', 'как', 'можно', 'неприметнее', 'мелочи']

X: ['вот', 'эти', 'мелочи', 'губят', 'всегда']
Y: ['эти', 'мелочи', 'губят', 'всегда', 'всё']

X: ['эти', 'мелочи', 'губят', 'всегда', 'всё']
Y: ['мелочи', 'губят', 'всегда', 'всё', 'идти']

X: ['мелочи', 'губят', 'всегда', 'всё', 'идти']
Y: ['губят', 'всегда', 'всё', 'идти', 'ему']

X: ['губят', 'всегда', 'всё', 'идти', 'ему']
Y: ['всегда', 'всё', 'идти', 'ему', 'было']

X: ['всегда', 'всё', 'идти', 'ему', 'было']
Y: ['всё', 'идти', 'ему', 'было', 'немного']

X: ['время', 'сам', 'еще', 'верил', 'этим']
Y: ['сам', 'еще', 'верил', 'этим', 'мечтам']

X: ['сам', 'еще', 'верил', 'этим', 'мечтам']
Y: ['еще', 'верил', 'этим', 'мечтам', 'своим']

X: ['еще', 'верил', 'этим', 'мечтам', 'своим']
Y: ['верил', 'этим', 'мечтам', 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 300) │ 19,903,200 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    570,368 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, None, 256) │    570,368 │ embedding[0][0],  │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 17,050,408 │ lstm_1[0][0]      │
│                     │ 66344)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 38,094,344 (145.32 MB)

 Trainable params: 18,191,144 (69.39 MB)

 Non-trainable params: 19,903,200 (75.92 MB)

Epoch 1/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 119s 156ms/step - acc: 0.0456 - loss: 8.3628 - val_acc: 0.1212 - val_loss: 7.2647
Epoch 2/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 116s 156ms/step - acc: 0.2164 - loss: 6.1418 - val_acc: 0.2801 - val_loss: 5.7118
Epoch 3/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 115s 156ms/step - acc: 0.3443 - loss: 4.6067 - val_acc: 0.3602 - val_loss: 4.9546
Epoch 4/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 115s 156ms/step - acc: 0.4652 - loss: 3.5128 - val_acc: 0.4172 - val_loss: 4.4533
Epoch 5/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 116s 157ms/step - acc: 0.5829 - loss: 2.6763 - val_acc: 0.4681 - val_loss: 4.1207
Epoch 6/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 116s 156ms/step - acc: 0.6815 - loss: 2.0475 - val_acc: 0.5050 - val_loss: 3.8832
Epoch 7/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 115s 156ms/step - acc: 0.7569 - loss: 1.5924 - val_acc: 0.5317 - val_loss: 3.7388
Epoch 8/10
740/740 ━━━━━━━━━━━━━━━━━━━━ 115s 156ms/step - acc: 0.8042 - loss: 1.2700 - val_acc: 0.5468 - val_loss: 3.6513
Epoch 9/10
740/740 ━━━━━

In [ ]:
test_loss, test_acc = model.evaluate(X_test, Y_test)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

perplexity = np.exp(test_loss)
print(f"Test perplexity: {perplexity:.4f}")

185/185 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - acc: 0.5687 - loss: 3.5777
Test loss: 3.5777
Test accuracy: 0.5687
Test perplexity: 35.7917


In [ ]:
model = keras.models.load_model("../saved_results/model_2.keras")

In [ ]:
def predict_next_word(text, top_k=5):
    # чистим так же, как при подготовке данных
    text = text.lower()
    text = re.sub(r'[^а-яё\s]', ' ', text)
    words = text.split()
    words = [w for w in words if len(w) >= MIN_WORD_LEN]

    # берём последние 5 слов
    words = words[-(MIN_SEN_LEN - 1):]

    for word in words:
        if word not in vocab:
            print(f"Слова нет в vocab: {word}")
            return

    x = np.array([[vocab[word] for word in words]], dtype=np.int32)

    preds = model.predict(x, verbose=0)

    # последнее предсказание — следующее слово после последнего слова входа
    probs = preds[0, -1]

    best_ids = np.argsort(probs)[-top_k:][::-1]

    print("Вход:", words)
    print("Варианты следующего слова:")

    for idx in best_ids:
        print(index_to_word[int(idx)], float(probs[idx]))

In [ ]:
predict_next_word("родион романович стоял у двери", top_k=5)

Вход: ['родион', 'романович', 'стоял', 'двери']
Варианты следующего слова:
стол 0.08573497086763382
побежал 0.07858146727085114
лестницу 0.04344099387526512
двери 0.042235322296619415
лавке 0.03488888218998909


In [ ]:
predict_next_word("Тогда Раскольников ударил", top_k=5)

Вход: ['тогда', 'раскольников', 'ударил']
Варианты следующего слова:
ударил 0.12285023182630539
своей 0.08733104914426804
его 0.08352910727262497
себя 0.05547082796692848
него 0.040257859975099564
